# DeepSCF: Electron Density Prediction and Electronic Properties

DeepSCF predicts self-consistent electron density from atomic features, as described in [*Convolutional network learning of self-consistent electron density via grid-projected atomic fingerprints*](https://doi.org/10.1038/s41524-024-01433-0). This tutorial uses the pretrained **(R+S)×3 model** to predict densities and SIESTA to calculate electronic properties from them.

For **polyethylene**, we prepare the model inputs, predict the density, and compare energies, forces, and band structures with an initial atomic-density calculation and a converged self-consistent field (SCF) reference. We then apply the workflow to **single-stranded DNA**, using a supplied SCF reference to compare densities and the density of states (DOS). Throughout the tutorial, we use `sisl` to read SIESTA data and visualize structures and electronic properties.

## 1. Setup

Use a **CPU runtime** in Google Colab and run the cells in order.

### Download the repositories

Download DeepSCF and this tutorial into `/content`. DeepSCF provides the model and modified SIESTA source; the tutorial provides calculation inputs and analysis scripts.

In [ ]:
%%bash
set -e
cd /content
if [ ! -d DeepSCF ]; then
  git clone https://github.com/yhkimlab/DeepSCF.git
  git -C DeepSCF checkout --detach fe3c3c18052b51101e438dad2ffa06f94e77b5bb
fi

if [ ! -d tutorial ]; then
  git clone --depth 1 --branch main https://github.com/ryonggyulee/tutorial.git tutorial
fi


In [ ]:
%%bash
set -e
cd /content/DeepSCF
# Check the tutorial snapshot.
[ "$(git rev-parse HEAD)" = fe3c3c18052b51101e438dad2ffa06f94e77b5bb ]
test -s requirements.txt
test -s src/siesta-4.1-b4/Obj/gfortran-system.make
git log -1 --format='%h %cs %s'
test -s /content/tutorial/deepscf/01/src/polymer/scf/RUN.fdf
test -s /content/tutorial/deepscf/01/src/polymer/nscf/RUN.fdf
test -s /content/tutorial/deepscf/01/src/predict.py
echo "DeepSCF and tutorial inputs ready"


### Install the dependencies

Install the packages for model inference and analysis. We use `sisl` to read SIESTA files and create interactive plots, and Matplotlib for density maps and electronic spectra.

In [ ]:
%pip install -q torch h5py hydra-core==1.3.2 omegaconf==2.3.0 "sisl[viz]" matplotlib

In [ ]:
%matplotlib inline
import sys, subprocess
import torch, h5py, numpy, hydra, omegaconf
import sisl, sisl.viz

print('Python:', sys.version.split()[0], '| PyTorch:', torch.__version__, '| sisl:', sisl.__version__)

### Build SIESTA

Compile the modified SIESTA bundled with DeepSCF. This version writes the atomic feature grids needed by the model and reads a supplied density for the non-SCF calculation. Compilation may take a few minutes; the build log is saved to `/content/siesta-build.log`.

In [ ]:
%%bash
set -e
apt-get update -qq
apt-get install -y -qq build-essential binutils gfortran libblas-dev liblapack-dev
cd /content/DeepSCF/src/siesta-4.1-b4/Obj
sh ../Src/obj_setup.sh
cp gfortran-system.make arch.make
# Save the build log.
make -j2 > /content/siesta-build.log 2>&1 || { tail -80 /content/siesta-build.log; exit 1; }
test -x siesta
echo "SIESTA build OK"


Launch SIESTA without an input file to check that the executable starts. The missing-input error and nonzero exit code are expected in this check.

In [ ]:
import subprocess, tempfile

with tempfile.TemporaryDirectory() as workdir:
    result = subprocess.run(
        ['/content/DeepSCF/src/siesta-4.1-b4/Obj/siesta'],
        stdin=subprocess.DEVNULL, capture_output=True, text=True,
        cwd=workdir, timeout=15,
    )
print(result.stdout)
print(result.stderr)
print('Exit code:', result.returncode, '(no input supplied)')


## 2. Reference SCF Calculation

Run an SCF calculation for polyethylene to obtain the reference density and electronic properties. The same run also produces the atomic feature grids used to prepare the model inputs.

Copy the supplied inputs into the working directory, `/content/results/polymer`. Use a fresh working directory when repeating the calculation: the runner stops if a previous `stdout.txt` exists.

In [ ]:
from pathlib import Path
import sys, shutil

ROOT = Path('/content/DeepSCF')
SRC = Path('/content/tutorial/deepscf/01/src')
PYTHON = sys.executable
SIESTA = ROOT / 'src/siesta-4.1-b4/Obj/siesta'
WORK = Path('/content/results/polymer')
WORK.mkdir(parents=True, exist_ok=True)
SCF, NSCF = WORK / 'scf/polymer', WORK / 'nscf'
INITIAL = WORK / 'initial'
for folder, mode in [(SCF, 'scf'), (INITIAL, 'nscf'), (NSCF, 'nscf')]:
    shutil.copytree(SRC / 'polymer' / mode, folder, dirs_exist_ok=True)
print('Working directory:', WORK)

### Inspect the structure

The polyethylene cell contains **24 atoms (C₈H₁₆)** in a **20 × 20 × 10 Å** cell. The chain is periodic along **z**, with vacuum in the transverse directions.

Use `sisl.get_sile(...).read_geometry()` to read `STRUCT.fdf`, then `geometry.plot()` to display the chain and cell boundaries with the Plotly backend.

In [ ]:
geometry = sisl.get_sile(str(SCF / 'STRUCT.fdf')).read_geometry()
print('Number of atoms:', len(geometry))
print('Cell vectors (Angstrom):\n', geometry.cell)

geometry.plot(
    axes='xyz',
    show_cell='box',
    backend='plotly'
).show()

### Review the SCF inputs

`RUN.fdf` defines the calculation settings and reads the structure from `STRUCT.fdf`. The calculation uses PBE, a double-zeta polarized (DZP) basis, and a **1 × 1 × 8 k-point mesh** at fixed atomic positions.

| Setting | Purpose |
|---|---|
| `MeshSizes 80 80 40` | Use a 0.25 Å grid spacing, matching the pretrained model. |
| `SaveRho T` | Save the converged electron density in `Molecular.RHO`. |
| `SaveIonicCharge T` | Save the ionic charge grid used as a model input. |
| `MaxSCFIterations 300`, `SCF.MustConverge T` | Allow up to 300 SCF iterations and require convergence. |

The full input and structure files are displayed below.

In [ ]:
from IPython.display import HTML, display
from html import escape

def show_input(path):
    # Render the full file without a scrolling output area.
    display(HTML(f'<h4>{escape(path.name)}</h4>'
                 f'<pre style="max-height: none; overflow: visible; white-space: pre-wrap;">'
                 f'{escape(path.read_text())}</pre>'))

for name in ['RUN.fdf', 'STRUCT.fdf']:
    show_input(SCF / name)


### Run the SCF calculation

Run SIESTA to obtain the converged density, total energy, atomic forces, and band structure. These provide the reference for the later comparisons. SCF iterations appear below; full logs are saved in the calculation folder.

In [ ]:
subprocess.run([PYTHON, '-u', str(SRC / 'run_siesta.py'), str(SCF),
                '--siesta', str(SIESTA)], check=True)

### Plot the electron density

Compare the initial atomic density—the superposition of neutral atomic densities—with the converged SCF density. The `plot_density` helper reads the grids with `sisl` and uses Matplotlib to plot their integrals along **y**, in **e/Å²**.

The **relative density error** is calculated over the full 3D grid:

$$
\epsilon_\rho = 100\,\frac{\sum_i |\rho_i - \rho_i^{\mathrm{SCF}}|}{\sum_i \rho_i^{\mathrm{SCF}}}.
$$

Here, $i$ indexes grid points, and the error is expressed as a percentage. The integrated SCF charge can be checked against the **48 valence electrons** in C₈H₁₆.

In [ ]:
sys.path.insert(0, str(SRC))
from analysis import plot_density, compare_observables

plot_density(initial=SCF / 'rhoatom', scf=SCF / 'Molecular.RHO');

### Plot the deformation density

The deformation density, $\Delta\rho = \rho_{\mathrm{SCF}} - \rho_{\mathrm{init}}$, measures the change from the initial atomic density. `SaveDeltaRho T` writes this field to `Molecular.DRHO`.

Read the grid with `sisl`, attach the atomic geometry with `set_geometry()`, and use `dr.plot()` to display interactive 3D isosurfaces. Orange marks positive values (electron accumulation), and blue marks negative values (depletion). `isovalue` sets the density threshold in **e/Å³**.

In [ ]:
dr_geometry = sisl.get_sile(str(SCF / 'STRUCT.fdf')).read_geometry()
dr = sisl.get_sile(str(SCF / 'Molecular.DRHO')).read_grid()
dr.set_geometry(dr_geometry)

isovalue = 0.08  # e/Angstrom^3
dr.plot(
    axes='xyz',
    interp=(3, 3, 3),
    smooth=True,
    isos=[
        {'val': isovalue, 'color': 'orange', 'opacity': 0.6, 'name': 'Electron accumulation'},
        {'val': -isovalue, 'color': 'royalblue', 'opacity': 0.6, 'name': 'Electron depletion'},
    ],
    plot_geom=True,
    show_cell='box',
    backend='plotly',
).show()


### Plot the band structure

Use `sisl` to read `Molecular.bands` and Matplotlib to plot the bands along $\Gamma$–$X$, where $X = (0, 0, 1/2)$ in reciprocal-lattice coordinates lies along the polymer axis.

The 48 valence electrons fill **24 doubly occupied bands**. Display the 12 occupied and 12 unoccupied bands nearest the gap, with energies measured from the **SCF valence-band maximum** on this path. The **band gap** is the difference between the lowest conduction-band energy and the highest valence-band energy among the sampled points.

In [ ]:
import matplotlib.pyplot as plt

reader = sisl.get_sile(str(SCF / "Molecular.bands"))
_, k, bands = reader.read_data()
bands = bands[:, 0, :] + reader.read_fermi_level()
nocc = 24
vbm = bands[:, nocc - 1].max()
gap = bands[:, nocc].min() - vbm
print(f"SCF gap along sampled Gamma-X path: {gap:.6f} eV")

fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)
ax.plot(k, bands[:, nocc - 12:nocc + 12] - vbm, "k-", lw=1.2)
ax.axhline(0, color="0.6", lw=0.5)
ax.set(xticks=[k[0], k[-1]], xticklabels=[r"$\Gamma$", "X"],
       xlim=(k[0], k[-1]), ylabel=r"$E-E_{\mathrm{VBM}}^{\mathrm{SCF}}$ (eV)")
plt.show()

## 3. Model Input Preparation

The model takes three grid channels: **ionic charge, initial atomic density, and orbital overlap**. The converged SCF density is stored separately as a reference for error evaluation; it is not a network input.

### Convert the SIESTA grids

Use `1.convert.py` to convert the SIESTA grids into HDF5 files. Each file contains a `value` array with shape `(80, 80, 40)`, preserving the original grid values.

| SIESTA file | HDF5 file under `raw/` | Quantity |
|---|---|---|
| `Molecular.RHO` | `target/polymer.h5` | SCF reference density |
| `Molecular.IOCH` | `feature1/polymer.h5` | Ionic charge density |
| `rhoatom` | `feature2/polymer.h5` | Initial atomic electron density |
| `rhoover` | `feature3/polymer.h5` | Orbital-overlap feature |

In [ ]:
subprocess.run([PYTHON, str(ROOT / 'tutorial/1.preprocess/1.convert.py'),
                str(WORK / 'scf'), str(WORK / 'raw')], check=True)

channels = ['target', 'feature1', 'feature2', 'feature3']
for channel in channels:
    path = WORK / 'raw' / channel / 'polymer.h5'
    with h5py.File(path, 'r') as data:
        values = data['value'][:]
        print(f'{path.relative_to(WORK)}: keys={list(data.keys())}, '
              f'shape={values.shape}, dtype={values.dtype}, '
              f'range=[{values.min():.6g}, {values.max():.6g}]')


### Load the standardization parameters

Load the **mean** and **standard deviation** for each input channel from `scale_r_plus_s_x3.txt`. Prediction must use the same parameters as training.

In [ ]:
SCALE = ROOT / 'datasets/scale/scale_r_plus_s_x3.txt'
print(SCALE.read_text())

scale = {}
for line in SCALE.read_text().splitlines():
    if line.strip():
        key, value = line.split('=')
        scale[key.strip()] = float(value)


### Standardize and assemble the model inputs

`3.collect.py` applies **standardization** to each input channel by subtracting its training **mean** and dividing by its training **standard deviation**. It then stacks the standardized ionic-charge, atomic-density, and overlap grids, in that order, into the `feature` array of shape `(3, 80, 80, 40)`.

The resulting `dataset/polymer.h5` also stores the unchanged SCF density as `target`. Since the standardization parameters are fixed from training, the channel means and standard deviations printed for this structure need not be zero and one.

In [ ]:
subprocess.run([PYTHON, str(ROOT / 'tutorial/1.preprocess/3.collect.py'),
                str(WORK / 'raw'), str(SCALE),
                str(WORK / 'dataset')], check=True)

with h5py.File(WORK / 'dataset/polymer.h5', 'r') as dataset:
    feature = dataset['feature'][:]
    target = dataset['target'][:]
    print('HDF5 datasets:', list(dataset.keys()))
    print('feature:', feature.shape, '| target:', target.shape)

for i, channel in enumerate(channels[1:]):
    with h5py.File(WORK / 'raw' / channel / 'polymer.h5', 'r') as data:
        raw = data['value'][:]
    expected = (raw - scale[f'mean{i + 2}']) / scale[f'std{i + 2}']
    numpy.testing.assert_allclose(feature[i], expected)
    print(f'feature[{i}] ({channel}): mean={feature[i].mean():.6f}, '
          f'std={feature[i].std():.6f}')

with h5py.File(WORK / 'raw/target/polymer.h5', 'r') as data:
    numpy.testing.assert_array_equal(target, data['value'][:])
print('Verified: three normalized features and an unchanged SCF target.')


## 4. Electron Density Prediction

### Download the pretrained model

Download the pretrained weights for the **(R+S)×3 model** from the model archive (about 486 MB).

In [ ]:
%%bash
set -euo pipefail
cd /content/DeepSCF
if [ ! -s pretrained/r_plus_s_x3/model.pt ]; then
  curl -fL --retry 3 https://github.com/yhkimlab/DeepSCF/releases/download/DeepSCF_v1/pretrained-models-v1.tar.gz \
    | tar --no-same-owner -xz pretrained/r_plus_s_x3/model.pt
fi
test -s pretrained/r_plus_s_x3/model.pt


### Predict the electron density

Run the model on CPU using the prepared input channels. The predicted density is saved as `predict/polymer.RHO` on the same cell and grid as the SCF reference.

In [ ]:
subprocess.run([PYTHON, str(SRC / 'predict.py'), str(ROOT), str(WORK)], check=True)
PREDICTED = WORK / 'predict/polymer.RHO'

### Compare the electron densities

Use `plot_density` to read the initial, predicted, and SCF density grids with `sisl` and display their y-integrated maps on a common color scale. The final panel shows the projected difference $\rho_{\mathrm{ML}} - \rho_{\mathrm{SCF}}$ on a separate color scale centered at zero.

For both the initial and predicted densities, the **relative density error** is calculated over the full 3D grid using the SCF density as the reference.

In [ ]:
plot_density(initial=SCF / 'rhoatom', ml=PREDICTED, scf=SCF / 'Molecular.RHO');

### Compare the deformation densities

Use `sisl` to read the density grids and subtract the same initial atomic density from the prediction and SCF reference. Display their isosurfaces with `Grid.plot()` in two Plotly panels: DeepSCF on the left and SCF on the right.

Both panels use a threshold of **0.08 e/Å³**. Orange shows accumulation relative to the atomic density, and blue shows depletion.

In [ ]:
from analysis import density
from plotly.subplots import make_subplots

isovalue = 0.08
atomic = density(SCF / 'rhoatom')
geometry = sisl.get_sile(str(SCF / 'STRUCT.fdf')).read_geometry()
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'scene'}, {'type': 'scene'}]],
    subplot_titles=('DeepSCF', 'SCF'),
)

for col, path in enumerate((PREDICTED, SCF / 'Molecular.RHO'), start=1):
    grid = density(path)
    grid.grid -= atomic.grid
    grid.set_geometry(geometry)
    plot = grid.plot(
        axes='xyz', interp=(3, 3, 3), smooth=True,
        isos=[
            {'val': isovalue, 'color': 'orange', 'opacity': 0.6},
            {'val': -isovalue, 'color': 'royalblue', 'opacity': 0.6},
        ],
        plot_geom=True, geom_kwargs={'show_bonds': True},
        show_cell='box', backend='plotly',
    )
    for trace in plot.data:
        fig.add_trace(trace, row=1, col=col)

fig.update_scenes(aspectmode='data')
fig.update_layout(showlegend=False)
fig.show()


## 5. Electronic Property Calculations

Perform separate non-SCF calculations from the initial atomic density and the predicted density, then compare their energies, forces, and bands with the converged SCF reference. All three cases use the same geometry, pseudopotentials, basis, grid, and k points.

### Prepare the non-SCF inputs

Copy each density to `In.RHO` in its calculation folder. The following settings control the non-SCF step:

- `SCF.Read.Charge T`: read the supplied density from `In.RHO`.
- `MaxSCFIterations 1`: perform one SCF iteration.
- `SCF.Mix.First F`: disable mixing on the first iteration.
- `SCF.MustConverge F`: allow the calculation to finish without SCF convergence.

`DM.UseSaveDM F` prevents reuse of a saved density matrix.

In [ ]:
for folder, rho in [(INITIAL, SCF / 'rhoatom'), (NSCF, PREDICTED)]:
    shutil.copy(rho, folder / 'In.RHO')
    print('Density input:', rho, '->', folder / 'In.RHO')
show_input(NSCF / 'RUN.fdf')


### Run the non-SCF calculations

Run SIESTA once for each input density. SIESTA constructs the Hamiltonian from that density, diagonalizes it, and uses the resulting density matrix to calculate the final energy and forces. The results therefore correspond to one SIESTA iteration starting from each supplied density.

An `SCF_NOT_CONV` message is expected because each calculation stops after one iteration.

In [ ]:
for label, folder in [('rho_init', INITIAL), ('rho_ML', NSCF)]:
    print(label)
    subprocess.run([PYTHON, '-u', str(SRC / 'run_siesta.py'), str(folder),
                    '--siesta', str(SIESTA), '--non-scf'], check=True)


### Compare the energies and forces

The table compares calculations using the initial density, the predicted density, and the converged SCF reference. Density errors use the supplied grids read with `sisl`; energies and forces come from the corresponding SIESTA outputs.

- **Energy error:** absolute difference from the SCF total energy, divided by the number of atoms, in meV/atom.
- **Force component RMSE:** root mean square of the errors in all Cartesian force components, in eV/Å.
- **Maximum force-vector error:** largest magnitude of the force difference for any atom, in eV/Å.

The maximum force magnitude is the largest force on any atom in each calculation. SCF forces need not vanish because the atomic positions have not been relaxed.

In [ ]:
from IPython.display import Markdown, display
summary = compare_observables(initial=INITIAL, ml=NSCF, scf=SCF)
display(Markdown(summary))

### Visualize the atomic forces

Use `sisl` to read the geometry and forces with `read_geometry()` and `read_force()`, then pass the force vectors to `geometry.plot()` as arrows. The panels show the initial-density, DeepSCF, and SCF calculations from left to right.

Red arrows indicate force directions and magnitudes. All panels use the same `force_scale`; increasing it enlarges the arrows without changing the underlying forces.

In [ ]:
from plotly.subplots import make_subplots

force_scale = 2.0
force_titles = ('Initial density', 'DeepSCF', 'SCF')
fig = make_subplots(
    rows=1, cols=3,
    specs=[[{'type': 'scene'} for _ in force_titles]],
    subplot_titles=force_titles,
)

for col, directory in enumerate((INITIAL, NSCF, SCF), start=1):
    fdf = sisl.get_sile(str(directory / 'RUN.fdf'))
    geometry = fdf.read_geometry()
    forces = fdf.read_force()
    if forces is None:
        raise FileNotFoundError(f'No force output found in {directory}. Run SIESTA first.')
    forces = numpy.asarray(forces)
    if forces.shape != (len(geometry), 3) or not numpy.isfinite(forces).all():
        raise ValueError(f'Invalid atomic forces in {directory}: {forces.shape}')
    plot = geometry.plot(
        axes='xyz', show_cell='box', show_bonds=True,
        arrows={
            'data': forces, 'color': 'red', 'width': 4,
            'scale': force_scale,
        },
        backend='plotly',
    )
    for trace in plot.data:
        fig.add_trace(trace, row=1, col=col)

fig.update_scenes(aspectmode='data')
fig.show()


### Compare the band structures

Use `sisl` to read the three band structures and overlay them along $\Gamma$–$X$. `read_data()` returns energies relative to each file's Fermi level. Add that Fermi level back, then shift all curves to the **SCF valence-band maximum**.

Calculate the **band gap** for each case from the same sampled path.

In [ ]:
import matplotlib.pyplot as plt

nocc = 24
reader = sisl.get_sile(str(SCF / "Molecular.bands"))
_, k, reference = reader.read_data()
vbm = reference[:, 0, nocc - 1].max() + reader.read_fermi_level()
fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)

for label, folder, style in [("Initial density", INITIAL, "b:"),
                             ("DeepSCF", NSCF, "r--"),
                             ("SCF", SCF, "k-")]:
    reader = sisl.get_sile(str(folder / "Molecular.bands"))
    _, k, bands = reader.read_data()
    bands = bands[:, 0, :] + reader.read_fermi_level()
    ax.plot(k, bands[:, nocc - 12:nocc + 12] - vbm, style, lw=1.2)
    ax.plot([], [], style, label=label)
    gap = bands[:, nocc].min() - bands[:, nocc - 1].max()
    print(f"{label} gap along sampled Gamma-X path: {gap:.6f} eV")

ax.axhline(0, color="0.6", lw=0.5)
ax.set(xticks=[k[0], k[-1]], xticklabels=[r"$\Gamma$", "X"],
       xlim=(k[0], k[-1]), ylabel=r"$E-E_{\mathrm{VBM}}^{\mathrm{SCF}}$ (eV)")
ax.legend()
plt.show()

## 6. DNA Example

Apply the same pretrained model to single-stranded DNA with **328 atoms** in a **40 × 40 × 32 Å** cell. We use a precomputed SCF reference because a full SCF calculation is time-consuming.

### Load the structure and SCF reference

Copy the supplied inputs and SCF results into the DNA working directory. Use `sisl` to read the structure and reference density, then display the structure with `geometry.plot()`.

In [ ]:
from analysis import density, plot_density_slice, plot_density_error

DNA_WORK = Path('/content/results/dna_single')
DNA_SCF = DNA_WORK / 'scf/dna_single'
DNA_NSCF = DNA_WORK / '2.nscf'
shutil.copytree(SRC / 'dna_single/scf', DNA_SCF, dirs_exist_ok=True)
shutil.copytree(SRC / 'dna_single/2.nscf', DNA_NSCF, dirs_exist_ok=True)

geometry = sisl.get_sile(str(DNA_SCF / 'STRUCT.fdf')).read_geometry()
reference = density(DNA_SCF / 'dna_single.RHO')
log = (DNA_SCF / 'stdout.txt').read_text()
print(next(line for line in log.splitlines() if 'SCF cycle converged' in line))
print('Atoms:', len(geometry), '| Grid:', reference.shape)
print('Integrated SCF charge: %.4f electrons' % (reference.grid.sum(dtype=numpy.float64) * reference.dvolume))
geometry.plot(axes='xyz', show_cell='box', backend='plotly').show()

### Predict the electron density

Use the saved ionic-charge, atomic-density, and overlap grids with the same model and normalization parameters as in the polyethylene example. The DNA grid is **160 × 160 × 128**, retaining the **0.25 Å** spacing. The `--prepare` option assembles the model inputs before prediction.

In [ ]:
subprocess.run([PYTHON, '-u', str(SRC / 'predict.py'), str(ROOT), str(DNA_WORK),
                '--case', 'dna_single', '--label', 'dna_single', '--prepare'], check=True)
DNA_PREDICTED = DNA_WORK / 'predict/dna_single.RHO'

### Compare the electron densities

The `plot_density_slice` helper reads the grids with `sisl` and displays an x–z slice at `y_slice` using Matplotlib. Both panels share the same color scale, in **e/Å³**. The **relative density error** is calculated over the full 3D grid.

The `plot_density_error` helper uses `sisl` and Plotly to display isosurfaces of $\rho_{\mathrm{ML}} - \rho_{\mathrm{SCF}}$. Orange marks positive errors and blue marks negative errors; `isovalue` sets their magnitude in **e/Å³**.

In [ ]:
y_slice = 20.0  # Å; choose a plane through the DNA.
plot_density_slice(ml=DNA_PREDICTED, scf=DNA_SCF / 'dna_single.RHO', y=y_slice);

In [ ]:
isovalue = 0.05  # e/Å³
plot_density_error(ml=DNA_PREDICTED, scf=DNA_SCF / 'dna_single.RHO',
                   structure=DNA_SCF / 'STRUCT.fdf', isovalue=isovalue);

### Run the non-SCF calculation

Copy the predicted density to `In.RHO` and run one SIESTA iteration using the supplied non-SCF input. The geometry, basis, grid, and k points match the SCF reference.

In [ ]:
shutil.copyfile(DNA_PREDICTED, DNA_NSCF / 'In.RHO')
subprocess.run([PYTHON, '-u', str(SRC / 'run_siesta.py'), str(DNA_NSCF),
                '--siesta', str(SIESTA), '--non-scf', '--label', 'dna_single', '--dos'], check=True)

### Compare the density of states

Use `sisl` to read eigenvalues from `.EIG` and k-point weights from `.KP`, then calculate the DOS with `sisl.physics.electron.DOS`. Weight each k-point contribution and multiply by two for spin degeneracy. Matplotlib displays the predicted-density and SCF results in **states/eV/cell**.

Both curves use the **SCF Fermi level** as the energy zero and the same **Gaussian broadening**, set by `smearing=0.10` eV.

In [ ]:
import matplotlib.pyplot as plt
from sisl.physics.electron import DOS
from sisl.physics.distribution import get_distribution

energy = numpy.linspace(-8, 5, 1301)
gaussian = get_distribution("gaussian", smearing=0.10)
reference = sisl.get_sile(str(DNA_SCF / "dna_single.EIG"))
fermi = reference.read_fermi_level()
fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)

for label, folder, style in [("SCF", DNA_SCF, "k-"),
                             ("DeepSCF", DNA_NSCF, "r--")]:
    reader = sisl.get_sile(str(folder / "dna_single.EIG"))
    eigenvalues = reader.read_data()[0] + reader.read_fermi_level() - fermi
    weights = sisl.get_sile(str(folder / "dna_single.KP")).read_data()[1]
    dos = 2 * sum(w * DOS(energy, eig, distribution=gaussian)
                  for w, eig in zip(weights, eigenvalues))
    ax.plot(energy, dos, style, label=label)

ax.axvline(0, color="0.6", lw=0.5)
ax.set(xlabel=r"$E-E_F$ (eV)",
       ylabel="DOS (states/eV/cell)", xlim=(-8, 5))
ax.legend()
plt.show()